# Imports and Initial Configuration

In [1]:
import json
import time
from pydantic import BaseModel, Field
from ollama import chat
from pathlib import Path
import torch

# ----- GLOBAL VARIABLES -----
model = "llama3:latest" # LLM model used for parameter extraction
ratio_queries = 1 # Ratio of the dataset to be processed (e.g., 1 = 100%, 0.1 = 10%)
queries_type = "queries_param_all" # Options: 

# ----- FILE PATHS -----
script_folder = Path().absolute()
tools_dir_path = script_folder.parent / 'input' / 'documentation' / 'manifest_tools'
queries_list_path = script_folder.parent / 'input' / 'param' / f'{queries_type}.json'
results_path = script_folder / 'output' / f'extraction_only_{queries_type}.json'


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

/home/luciacev/anaconda3/envs/paul_agent/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


PyTorch version: 2.6.0+cu124
CUDA available: True


# LLM Parameter Extraction

In [2]:
class ParameterExtraction(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning for extracting these parameters from the user's prompt.")
    parameters: dict = Field(description="Extracted parameters as a key-value dictionary. Use the exact parameter names from the tool schema.")


def extract_parameters(user_prompt, selected_tool_dict, model_name):
    tool_formatted = json.dumps(selected_tool_dict, indent=2)
    
    system_prompt = f"""You are an expert Parameter Extraction Agent for medical and dental imaging tools.
    Your task is to analyze the user's request and extract the required parameters according to the selected tool's schema.
    
    === SELECTED TOOL SCHEMA ===
    {tool_formatted}
    
    Carefully read the user's prompt and extract the appropriate values for the tool's parameters.
    Output strictly matching the JSON schema. If an optional parameter is missing, do not invent it.
    """
    
    try:
        response = chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            format=ParameterExtraction.model_json_schema(),
            options={"temperature": 0.0},
        )
        return ParameterExtraction.model_validate_json(response.message.content)
    except Exception as e:
        return ParameterExtraction(parameters={}, reasoning=f"Error: {str(e)}")

# Data Initialization and Caching Setup

In [3]:
# ----- LOAD FILES -----
tools_list = []
for file_path in tools_dir_path.glob('*.json'):
    with open(file_path, 'r', encoding='utf-8') as f:
        tools_list.append(json.load(f))

with open(queries_list_path, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

print(f"Loaded {len(tools_list)} tools from manifest_tools and {len(queries_list)} queries.")

Loaded 23 tools from manifest_tools and 250 queries.


# Main Loop

In [4]:
# ----- METRICS INITIALIZATION -----
total_extraction_latency = 0
results_detail = []
total_perfect_matches = 0
total_with_expected_params = 0
total_expected_individual_params = 0
total_correct_individual_params = 0

# Calculate the total number of queries to process based on the ratio
total_queries = int(len(queries_list) * ratio_queries)
queries_to_run = queries_list[:total_queries]

# Iterate over each query in the dataset
for index, q in enumerate(queries_to_run, start=1):
    if isinstance(q, list):
        prompt = q[0]
        expected_tool = q[1]
        expected_params = None
    else:
        prompt = q['query']
        expected_tool = q['expected_tool']
        expected_params = q.get('expected_params')
        
    print(f"\n{'='*65}")
    print(f"Prompt ({index}/{total_queries}) : '{prompt}'")
    print(f"Expected Tool: {expected_tool}")
    print(f"{'-'*65}")

    # Identify the correct tool details to pass to extractor
    selected_tool_dict = next((t for t in tools_list if t.get("name") == expected_tool), None)
    
    extraction = None
    extraction_latency = 0
    
    if selected_tool_dict:
        t0_ext = time.time()
        extraction = extract_parameters(prompt, selected_tool_dict, model)
        t1_ext = time.time()
        extraction_latency = t1_ext - t0_ext
        total_extraction_latency += extraction_latency
        
        # Check extraction accuracy
        match_sign = ""
        if expected_params is not None:
            total_with_expected_params += 1
            
            # Count individual parameter matches
            for k, v in expected_params.items():
                total_expected_individual_params += 1
                if extraction.parameters.get(k) == v:
                    total_correct_individual_params += 1
                    
            if extraction.parameters == expected_params:
                match_sign = "✅"
                total_perfect_matches += 1
            else:
                match_sign = "❌"
                
        print(f"PARAMETER EXTRACTION {match_sign}")
        print(f"Extracted Params: {json.dumps(extraction.parameters, indent=2)}")
        if expected_params is not None and match_sign != "✅":
            print("Expected Params: {")
            items = list(expected_params.items())
            for i, (k, v) in enumerate(items):
                emoji = "✅" if extraction.parameters.get(k) == v else "❌"
                comma = "," if i < len(items) - 1 else ""
                print(f'  "{k}": {json.dumps(v)}{comma} {emoji}')
            print("}")
        print(f"Latency: {extraction_latency:.2f} seconds")
        print(f"Reasoning: {extraction.reasoning[:150]}...")
    else:
        print(f"Tool '{expected_tool}' not found in tools list. Skipping extraction.")
        
    # Store detailed results for the current query
    results_detail.append({
        "prompt": prompt,
        "expected_tool": expected_tool,
        "expected_params": expected_params,
        "extracted_params": extraction.parameters if extraction else None,
        "extraction_latency": extraction_latency,
        "extraction_reasoning": extraction.reasoning if extraction else None
    })


Prompt (1/250) : 'Can you find the dots on the bone scans in /data/scans? Use the models in /models/ali and save them to /output. My computer needs /tmp for a second and these are simple files, not DICOM.'
Expected Tool: ali_cbct
-----------------------------------------------------------------
PARAMETER EXTRACTION ❌
Extracted Params: {
  "input": "/data/scans",
  "dir_models": "/models/ali",
  "lm_type": "dots",
  "output_dir": "/output",
  "temp_fold": "/tmp",
  "DCMInput": false
}
Expected Params: {
  "input": "/data/scans", ✅
  "dir_models": "/models/ali", ✅
  "lm_type": "1,2,3", ❌
  "output_dir": "/output", ✅
  "temp_fold": "/tmp", ✅
  "DCMInput": false ✅
}
Latency: 5.73 seconds
Reasoning: extracting parameters from user prompt...

Prompt (2/250) : 'I need to mark teeth 8 and 9 on the picture at /data/tooth.stl. Use the /models/ios folder and save the result to /output/marks. Please write down what happened in /output/log.txt.'
Expected Tool: ali_ios
-----------------------------

# Metrics Calculation and Results Export

In [5]:
# ----- FINAL METRICS CALCULATION -----
avg_extraction_latency = total_extraction_latency / total_queries if total_queries > 0 else 0
query_accuracy = (total_perfect_matches / total_with_expected_params * 100) if total_with_expected_params > 0 else 0
param_accuracy = (total_correct_individual_params / total_expected_individual_params * 100) if total_expected_individual_params > 0 else 0

# Build the final summary dictionary
summary = {
    "mode": "parameter_extraction_only",
    "queries_type": queries_type,
    "model": model,
    "metrics": {
        "total_queries": total_queries,
        "avg_extraction_latency": round(avg_extraction_latency, 4),
        "total_with_expected_params": total_with_expected_params,
        "total_perfect_matches": total_perfect_matches,
        "query_accuracy_percentage": round(query_accuracy, 2),
        "total_expected_individual_params": total_expected_individual_params,
        "total_correct_individual_params": total_correct_individual_params,
        "param_accuracy_percentage": round(param_accuracy, 2)
    },
    "details": results_detail,
}

# ----- EXPORT RESULTS -----
# Ensure the output directory exists
results_path.parent.mkdir(parents=True, exist_ok=True)

# Save the benchmark results to a JSON file
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

# ----- FINAL SUMMARY OUTPUT TO TERMINAL -----
print("\n === BENCHMARK COMPLETED ===")
print(f"Total Queries Processed: {total_queries}")
print(f"Average Extract Latency: {avg_extraction_latency:.4f} sec")
if total_with_expected_params > 0:
    print(f"Tool Extraction Accuracy (all params perfect): {total_perfect_matches}/{total_with_expected_params} ({query_accuracy:.2f}%)")
if total_expected_individual_params > 0:
    print(f"Individual Param Extraction Accuracy: {total_correct_individual_params}/{total_expected_individual_params} ({param_accuracy:.2f}%)")
print(f"Results saved to: {results_path}")


 === BENCHMARK COMPLETED ===
Total Queries Processed: 250
Average Extract Latency: 2.0496 sec
Tool Extraction Accuracy (all params perfect): 127/250 (50.80%)
Individual Param Extraction Accuracy: 1022/1178 (86.76%)
Results saved to: /media/luciacev/Data/Paul_Agent/PARAM/output/extraction_only_queries_param_all.json
